# 05 — Hidden Markov Model inference

This notebook performs exploratory Hidden Markov Model (HMM) inference on frame-to-frame velocity sequences.

Input generated by `03_velocity_analysis.ipynb`:

```text
results/velocity/frame_to_frame_links.csv
```

The notebook performs:
- trajectory-wise sequence preparation,
- Gaussian HMM fitting on `log10(velocity)`,
- hidden mobility-state assignment,
- state ordering into slow / intermediate / fast states,
- transition matrix extraction,
- per-cell HMM-state summaries,
- HMM state-fraction and transition visualizations.

Important: HMM results are exploratory. Short sptPALM trajectories and tracking uncertainty can affect state inference, so results should be interpreted as mobility-state patterns rather than direct proof of molecular mechanisms.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from hmmlearn.hmm import GaussianHMM
except ImportError as error:
    raise ImportError(
        "hmmlearn is required for this notebook. "
        "Install it with: pip install hmmlearn"
    ) from error


plt.style.use("default")


CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RESULTS_DIR = PROJECT_DIR / "results"
VELOCITY_DIR = RESULTS_DIR / "velocity"
HMM_DIR = RESULTS_DIR / "hmm"

FIGURES_DIR = PROJECT_DIR / "figures"
HMM_FIGURES_DIR = FIGURES_DIR / "hmm"

for directory in [
    RESULTS_DIR,
    VELOCITY_DIR,
    HMM_DIR,
    FIGURES_DIR,
    HMM_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


LINKS_PATH = VELOCITY_DIR / "frame_to_frame_links.csv"

CONDITION_ORDER = [
    "control",
    "500uM_H2O2",
    "500uM_H2O2_30min",
]

CONDITION_LABELS = {
    "control": "Control",
    "500uM_H2O2": "500 µM H₂O₂",
    "500uM_H2O2_30min": "500 µM H₂O₂ 30 min",
}

N_HMM_STATES = 3
MIN_LINKS_PER_TRACK = 3
RANDOM_STATE = 42
N_ITER = 300

STATE_LABELS = {
    0: "slow",
    1: "intermediate",
    2: "fast",
}

print("Project directory:", PROJECT_DIR)
print("Links file:", LINKS_PATH)
print("Configuration loaded")


## 2. Load velocity links

In [ ]:
if not LINKS_PATH.exists():
    raise FileNotFoundError(
        f"Velocity links file was not found: {LINKS_PATH}. "
        "Run 03_velocity_analysis.ipynb first."
    )

links = pd.read_csv(LINKS_PATH)

required_columns = [
    "protein",
    "condition",
    "sample",
    "cell",
    "file_name",
    "track_id",
    "start_frame",
    "velocity_um_s",
]

missing_columns = [
    column for column in required_columns
    if column not in links.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(links.columns)}"
    )

links = links[
    np.isfinite(links["velocity_um_s"])
    & (links["velocity_um_s"] > 0)
].copy()

links["log10_velocity"] = np.log10(links["velocity_um_s"])

links = links.sort_values(
    ["protein", "sample", "cell", "condition", "track_id", "start_frame"]
).reset_index(drop=True)

print("Loaded links:", len(links))
print("Proteins:", sorted(links["protein"].dropna().unique()))

links.head()


## 3. Prepare HMM sequences

The HMM is fitted separately for each protein.  
Each trajectory is treated as an independent observation sequence.

The observed variable is:

```text
log10(frame-to-frame velocity)
```

Short trajectories with fewer than `MIN_LINKS_PER_TRACK` links are excluded from HMM fitting.


In [ ]:
def prepare_hmm_sequences(data, min_links_per_track=3):
    """Prepare concatenated HMM observation matrix and sequence lengths."""
    observations = []
    lengths = []
    index_order = []

    group_columns = [
        "condition",
        "sample",
        "cell",
        "file_name",
        "track_id",
    ]

    for _, group in data.groupby(group_columns, sort=False):
        group = group.sort_values("start_frame")

        if len(group) < min_links_per_track:
            continue

        values = group["log10_velocity"].to_numpy(dtype=float)

        if not np.all(np.isfinite(values)):
            continue

        observations.append(values.reshape(-1, 1))
        lengths.append(len(values))
        index_order.extend(group.index.tolist())

    if not observations:
        return None, [], []

    observations = np.vstack(observations)

    return observations, lengths, index_order


def order_hmm_states_by_mean_velocity(result_table):
    """Reorder HMM states from slow to fast based on mean log10 velocity."""
    state_order = (
        result_table.groupby("hmm_state_raw")["log10_velocity"]
        .mean()
        .sort_values()
        .index
    )

    state_map = {
        old_state: new_state
        for new_state, old_state in enumerate(state_order)
    }

    result_table["hmm_state"] = result_table["hmm_state_raw"].map(state_map)
    result_table["hmm_state_label"] = result_table["hmm_state"].map(STATE_LABELS)

    return result_table, state_map


## 4. Fit HMM models

In [ ]:
def fit_hmm_for_protein(data, protein, n_states=3):
    """Fit a Gaussian HMM to one protein and return model and classified links."""
    protein_data = data[data["protein"] == protein].copy()

    observations, lengths, index_order = prepare_hmm_sequences(
        protein_data,
        min_links_per_track=MIN_LINKS_PER_TRACK,
    )

    if observations is None:
        print(f"No valid sequences for {protein}")
        return None, pd.DataFrame(), {}

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=N_ITER,
        random_state=RANDOM_STATE,
    )

    model.fit(observations, lengths)

    states_raw = model.predict(observations, lengths)

    result = protein_data.loc[index_order].copy()
    result["hmm_state_raw"] = states_raw

    result, state_map = order_hmm_states_by_mean_velocity(result)

    return model, result, state_map


hmm_models = {}
hmm_state_maps = {}
hmm_results = []

for protein in sorted(links["protein"].dropna().unique()):
    print("Fitting HMM for:", protein)

    model, result, state_map = fit_hmm_for_protein(
        links,
        protein,
        n_states=N_HMM_STATES,
    )

    if model is None:
        continue

    hmm_models[protein] = model
    hmm_state_maps[protein] = state_map
    hmm_results.append(result)

hmm_links = pd.concat(hmm_results, ignore_index=True) if hmm_results else pd.DataFrame()

hmm_links_path = HMM_DIR / "links_hmm_states.csv"
hmm_links.to_csv(hmm_links_path, index=False)

print("Saved:", hmm_links_path)

hmm_links.head()


## 5. HMM model parameters

In [ ]:
parameter_rows = []

for protein, model in hmm_models.items():
    result = hmm_links[hmm_links["protein"] == protein].copy()

    if result.empty:
        continue

    ordered_states = (
        result.groupby("hmm_state")["log10_velocity"]
        .mean()
        .sort_index()
    )

    for state in range(N_HMM_STATES):
        state_data = result[result["hmm_state"] == state]

        parameter_rows.append({
            "protein": protein,
            "hmm_state": state,
            "state_label": STATE_LABELS.get(state, f"state_{state}"),
            "mean_log10_velocity": state_data["log10_velocity"].mean(),
            "median_velocity_um_s": state_data["velocity_um_s"].median(),
            "mean_velocity_um_s": state_data["velocity_um_s"].mean(),
            "n_links": len(state_data),
        })

hmm_parameters = pd.DataFrame(parameter_rows)

hmm_parameters_path = HMM_DIR / "hmm_state_parameters.csv"
hmm_parameters.to_csv(hmm_parameters_path, index=False)

print("Saved:", hmm_parameters_path)

hmm_parameters


## 6. Transition matrices

In [ ]:
def reorder_transition_matrix(model, state_map):
    """Reorder transition matrix according to slow-to-fast state map."""
    inverse_map = {
        new_state: old_state
        for old_state, new_state in state_map.items()
    }

    ordered_old_states = [inverse_map[i] for i in range(len(inverse_map))]

    transition_matrix = model.transmat_
    transition_matrix = transition_matrix[np.ix_(ordered_old_states, ordered_old_states)]

    return transition_matrix


transition_rows = []

for protein, model in hmm_models.items():
    transition_matrix = reorder_transition_matrix(
        model,
        hmm_state_maps[protein],
    )

    for from_state in range(transition_matrix.shape[0]):
        for to_state in range(transition_matrix.shape[1]):
            transition_rows.append({
                "protein": protein,
                "from_state": from_state,
                "from_state_label": STATE_LABELS.get(from_state, f"state_{from_state}"),
                "to_state": to_state,
                "to_state_label": STATE_LABELS.get(to_state, f"state_{to_state}"),
                "transition_probability": transition_matrix[from_state, to_state],
            })

transition_table = pd.DataFrame(transition_rows)

transition_table_path = HMM_DIR / "hmm_transition_matrices.csv"
transition_table.to_csv(transition_table_path, index=False)

print("Saved:", transition_table_path)

transition_table.head()


## 7. Per-cell HMM-state summary

In [ ]:
cell_summary_rows = []

if not hmm_links.empty:
    for keys, group in hmm_links.groupby(
        ["protein", "condition", "sample", "cell", "file_name"],
        observed=True,
    ):
        protein, condition, sample, cell, file_name = keys
        n_links = len(group)

        state_counts = group["hmm_state_label"].value_counts()

        cell_summary_rows.append({
            "protein": protein,
            "condition": condition,
            "sample": sample,
            "cell": cell,
            "file_name": file_name,
            "n_links": n_links,
            "slow_fraction": state_counts.get("slow", 0) / n_links,
            "intermediate_fraction": state_counts.get("intermediate", 0) / n_links,
            "fast_fraction": state_counts.get("fast", 0) / n_links,
            "slow_percent": 100 * state_counts.get("slow", 0) / n_links,
            "intermediate_percent": 100 * state_counts.get("intermediate", 0) / n_links,
            "fast_percent": 100 * state_counts.get("fast", 0) / n_links,
        })

hmm_cell_summary = pd.DataFrame(cell_summary_rows)

hmm_cell_summary_path = HMM_DIR / "cell_level_hmm_state_summary.csv"
hmm_cell_summary.to_csv(hmm_cell_summary_path, index=False)

print("Saved:", hmm_cell_summary_path)

hmm_cell_summary.sort_values(
    ["protein", "sample", "cell", "condition"],
    na_position="last",
)


## 8. Plotting functions

In [ ]:
def setup_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)


def plot_hmm_state_fractions(protein):
    data = hmm_links[hmm_links["protein"] == protein].copy()

    if data.empty:
        print(f"No HMM data for {protein}")
        return

    counts = (
        data.groupby(["condition", "hmm_state_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(CONDITION_ORDER)
        .fillna(0)
    )

    fractions = counts.div(counts.sum(axis=1), axis=0)

    state_order = ["slow", "intermediate", "fast"]

    fig, ax = plt.subplots(figsize=(7, 5))
    bottom = np.zeros(len(fractions))

    for state in state_order:
        if state not in fractions.columns:
            continue

        ax.bar(
            range(len(fractions)),
            fractions[state].values,
            bottom=bottom,
            label=state,
        )
        bottom += fractions[state].values

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("Fraction of links")
    ax.set_title(f"{protein}: HMM mobility-state fractions")
    ax.legend(frameon=False)

    setup_axis(ax)
    plt.tight_layout()

    output_path = HMM_FIGURES_DIR / f"{protein}_hmm_state_fractions.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_hmm_fast_fraction_per_cell(protein):
    data = hmm_cell_summary[
        hmm_cell_summary["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No cell-level HMM data for {protein}")
        return

    data["condition"] = pd.Categorical(
        data["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    data = data.sort_values(["sample", "cell", "condition"])
    x_map = {condition: i for i, condition in enumerate(CONDITION_ORDER)}

    fig, ax = plt.subplots(figsize=(7, 5))

    for (sample, cell), group in data.groupby(["sample", "cell"], observed=True):
        group = group.sort_values("condition")
        x = group["condition"].map(x_map).astype(float).to_numpy()
        y = group["fast_percent"].to_numpy()

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            alpha=0.55,
            label=f"S{sample} cell{cell}",
        )

    median_summary = (
        data.groupby("condition", observed=True)["fast_percent"]
        .median()
        .reindex(CONDITION_ORDER)
    )

    ax.plot(
        range(len(CONDITION_ORDER)),
        median_summary.values,
        marker="o",
        linewidth=4,
        color="black",
        label="group median",
    )

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("HMM fast-state links per cell, %")
    ax.set_title(f"{protein}: HMM fast-state fraction per cell")

    setup_axis(ax)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
    plt.tight_layout()

    output_path = HMM_FIGURES_DIR / f"{protein}_hmm_fast_fraction_per_cell.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_transition_matrix(protein):
    matrix_data = transition_table[
        transition_table["protein"] == protein
    ].copy()

    if matrix_data.empty:
        print(f"No transition matrix for {protein}")
        return

    matrix = matrix_data.pivot(
        index="from_state_label",
        columns="to_state_label",
        values="transition_probability",
    ).reindex(index=["slow", "intermediate", "fast"],
              columns=["slow", "intermediate", "fast"])

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(matrix.values, vmin=0, vmax=1)

    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns, rotation=20)
    ax.set_yticklabels(matrix.index)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(
                j,
                i,
                f"{matrix.values[i, j]:.2f}",
                ha="center",
                va="center",
            )

    ax.set_xlabel("To state")
    ax.set_ylabel("From state")
    ax.set_title(f"{protein}: HMM transition matrix")

    plt.colorbar(image, ax=ax, label="Transition probability")
    plt.tight_layout()

    output_path = HMM_FIGURES_DIR / f"{protein}_hmm_transition_matrix.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_hmm_state_distributions(protein):
    data = hmm_links[hmm_links["protein"] == protein].copy()

    if data.empty:
        print(f"No HMM state data for {protein}")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    for state_label in ["slow", "intermediate", "fast"]:
        state_values = data.loc[
            data["hmm_state_label"] == state_label,
            "log10_velocity",
        ].dropna().to_numpy()

        if len(state_values) == 0:
            continue

        ax.hist(
            state_values,
            bins=60,
            density=True,
            alpha=0.45,
            label=state_label,
        )

    ax.set_xlabel("log10 velocity, velocity in µm/s")
    ax.set_ylabel("Probability density")
    ax.set_title(f"{protein}: HMM state distributions")
    ax.legend(frameon=False)

    setup_axis(ax)
    plt.tight_layout()

    output_path = HMM_FIGURES_DIR / f"{protein}_hmm_state_distributions.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


## 9. Generate HMM figures

In [ ]:
proteins = sorted(hmm_links["protein"].dropna().unique())

for protein in proteins:
    plot_hmm_state_fractions(protein)
    plot_hmm_fast_fraction_per_cell(protein)
    plot_transition_matrix(protein)
    plot_hmm_state_distributions(protein)


## 10. Quick interpretation guide

- HMM assigns each frame-to-frame movement to a hidden mobility state.
- States are ordered by mean velocity and labeled as `slow`, `intermediate`, and `fast`.
- Transition matrices describe estimated probabilities of switching between mobility states.
- Because sptPALM trajectories are often short, HMM inference should be treated as exploratory.
- Cell-level summaries are preferable for condition comparison because individual links are not independent biological replicates.


## 11. Output files

This notebook generates:

```text
results/hmm/links_hmm_states.csv
results/hmm/hmm_state_parameters.csv
results/hmm/hmm_transition_matrices.csv
results/hmm/cell_level_hmm_state_summary.csv
figures/hmm/*_hmm_state_fractions.png
figures/hmm/*_hmm_fast_fraction_per_cell.png
figures/hmm/*_hmm_transition_matrix.png
figures/hmm/*_hmm_state_distributions.png
```

These outputs can be used as an advanced exploratory extension of the main diffusion and velocity analysis pipeline.
